# Paper figures and tables

Regenerates the data figures (Figs. 2–9) and Tables 4–5 of *Synthesizing Fast and Accurate Immediate-Response DRAM Latency Model with Agents* from the data in `results/`. Figures go to `results/figures/` (PDF/SVG/PNG), tables to `results/tables/`.

Model keys: `<model>_single_core` is the model after the 10 single-core iterations, `<model>_multicore` after 5 more multicore iterations; `fixedlat`, `md1`, `wmg1` and `mess` are the baselines. Core-cycle error is the absolute error of simulated CPU cycles against the cycle-level oracle; MAE<sub>Req</sub> is the mean absolute per-request latency error divided by the mean oracle read latency.

In [ ]:
import subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
import paper_figures as pf

ROOT = pf.ROOT
FIGURES, TABLES = ROOT / 'results/figures', ROOT / 'results/tables'
pf.style()

def show(render):
    render(FIGURES)
    display(Image(filename=str(FIGURES / (render.__name__.replace('_', '-') + '.png'))))

def data(name):
    return pd.read_csv(ROOT / 'results/paper' / f'{name}.csv')

## Fig. 2: Per-workload test errors

Core-cycle error (top) and MAE<sub>Req</sub> (bottom) on each of the 24 single-core test workloads. S*g*/*r* and T/*r* denote the Google workloads sierra (group *g*, region *r*) and tahoe (region *r*). Data: `results/paper/single-core-test.csv`.

In [ ]:
show(pf.fig2_per_workload)

## Fig. 3: Mean test error

Mean core-cycle error and MAE<sub>Req</sub> over the 24 test workloads. Data: `results/paper/single-core-test-means.csv` (the means of `single-core-test.csv`).

In [ ]:
show(pf.fig3_accuracy)
data('single-core-test-means')

## Fig. 4: Errors during the CHIA loop

Training and validation errors of the selected model after each iteration (lines) and of each iteration's candidate, accepted (dots) or rejected (crosses). Iteration 0 is the common seed model. Data: `results/paper/convergence.csv`.

In [ ]:
show(pf.fig4_convergence)

## Fig. 5: Simulation speedup

Speedup of each DRAM model over the cycle-level oracle, by traffic pattern, read ratio and request arrival interval; each point is the median of five repetitions. Each speedup is relative to the oracle measured on the same host (`host` column). Data: `results/paper/speed.csv`; all repetitions are in `results/diagnostics/dram-speed*.json`.

In [ ]:
show(pf.fig5_speed)
speed = data('speed').query('model != "oracle"')
speed.groupby('model').oracle_relative_speedup.apply(lambda s: s.prod() ** (1 / len(s))).rename('geometric-mean speedup (30 points)')

## Fig. 6: LLM usage

Recorded token usage (solid: input; hatched: output) and estimated API-equivalent cost of the 10 single-core iterations. † Gemini did not report its cache hit rate, so its cost spans 0% to 100% hit rate. Data: `results/paper/usage.csv`; per-round and per-phase usage is in `results/foundation/single-core-usage.json`.

In [ ]:
show(pf.fig6_usage)
data('usage')

## Fig. 7: gem5

Mean absolute elapsed-time error of the single-core models in gem5 (O3CPU) over the 6 SE and 9 FS workloads. Data: `results/paper/gem5.csv`.

In [ ]:
show(pf.fig7_gem5)

## Fig. 8: DRAM organization and queue depth

ChampSim accuracy on the 24 single-core test workloads after changing the DRAM bank organization and the controller queue depth, without retraining. Data: `results/paper/hardware.csv`; per-case results for all models, including the multicore models and the baselines, are in `results/extensions/`.

In [ ]:
show(pf.fig8_hardware)

## Fig. 9: Multicore

Errors averaged over the eight test mixes at four and eight cores. Open markers: models after the 10 single-core iterations; filled markers: after 5 more multicore iterations. Data: `results/paper/multicore.csv`.

In [ ]:
show(pf.fig9_multicore)
data('multicore')

## Table 4: GPT-6 Astra feature-removal ablation

The frozen GPT-6 Astra model with one mechanism removed per variant, on the 24 single-core test workloads. The paper lists the control and the three largest changes. Data: `results/foundation/astra-ablation/`.

In [ ]:
out = TABLES / 'table4'
subprocess.run([sys.executable, '-I', str(ROOT / 'scripts/astra_ablation/summarize.py'),
                '--run', str(ROOT / 'results/foundation/astra-ablation/run'),
                '--variants', str(ROOT / 'results/foundation/astra-ablation/variants/variants.json'),
                '--expected', str(ROOT / 'results/foundation/astra-ablation/expected_control.json'),
                '--out', str(out)], check=True, capture_output=True)
pd.read_csv(out / 'ablation_astra_sc_summary.csv')[['variant', 'description', 'core_err_pct', 'mae_over_L']]

## Table 5: Request-level multicore metrics

Paired MAE<sub>Req</sub> and the pairing-free Wasserstein-1 distance between each model's and the oracle's per-core read-latency distributions (divided by the mean oracle read latency), averaged over cores and the eight test mixes. Data: `results/extensions/multicore/metrics/`.

In [ ]:
out = TABLES / 'table5'
subprocess.run([sys.executable, '-I', str(ROOT / 'scripts/multicore_metrics/table5.py'), str(ROOT), str(out)],
               check=True, capture_output=True)
pd.read_csv(out / 'table5.csv')